# Paper 5.2 — OU process with a fixed non-invariant dictionary

OU process $dX_t=-X_t\,dt+\sqrt2\,dW_t$ with $\rho=N(0,1)$, the dictionary $\{1,\sin x,\cos x,\sin 2x,\cos 2x\}$
(not generator invariant, $\mathcal A\sin x=-x\cos x-\sin x\notin\mathcal V_N$), and horizon $T=1$.
Population matrices use 160 Gauss–Hermite nodes; $\mathcal K^T\cos x=e^{-(1-e^{-2T})/2}\cos(e^{-T}x)$ is exact.

This notebook reproduces

* $\Theta_N$, the eigenvalues of $A_N$, the leading operator $\mathcal L_{N,T}=\tfrac12\int_0^Te^{(T-s)A_N}D_Ne^{sA_N}\,ds$
  of Theorem 3.3, see (3.10), and its lower bound (3.11);
* Table 4: time-stepping errors $\|Q^n-e^{TA_N}\|$ of SDMD-1, SDMD-R, SDMD-2 in the $G$-weighted norm, the normalized
  distance of $((Q^{[2]})^n-e^{TA_N})/\tau$ from $\mathcal L_{N,T}$, and the errors against $\mathcal K^T\cos x$;
* Table 5: errors $\|\widehat Q^n-e^{TA_N}\|$ at $\Delta t=0.05$ with empirical matrices from $m$ i.i.d. samples
  (25 repetitions, seed 20260507), with percentile bands and the perturbations of the Galerkin matrices.

In [1]:
import math
import numpy as np
import scipy.linalg as sla
from scipy.linalg import expm, solve
from numpy.polynomial.hermite import hermgauss

SEED = 20260507
T = 1.0


def normal_quadrature(n=160, std=1.0):
    """Gauss-Hermite nodes and weights for the Gaussian law N(0, std^2)."""
    nodes, weights = hermgauss(n)
    return math.sqrt(2.0) * std * nodes, weights / math.sqrt(math.pi)


def g_weighted_op_norm(M, G):
    """Operator norm on V_N with Gram matrix G, i.e. ||R M R^{-1}||_2 with G = R^T R."""
    R = sla.cholesky(0.5 * (G + G.T), lower=False)
    Rinv = sla.solve_triangular(R, np.eye(R.shape[0]), lower=False)
    return float(np.linalg.norm(R @ M @ Rinv, 2))


def trig_dictionary(x):
    """psi = {1, sin x, cos x, sin 2x, cos 2x} and its derivatives up to order four, evaluated at x."""
    psi, d1, d2, d3, d4 = (np.zeros((x.size, 5)) for _ in range(5))
    psi[:, 0] = 1.0
    for col, (k, kind) in enumerate([(1, "sin"), (1, "cos"), (2, "sin"), (2, "cos")], start=1):
        s, c = np.sin(k * x), np.cos(k * x)
        if kind == "sin":
            psi[:, col], d1[:, col], d2[:, col], d3[:, col], d4[:, col] = s, k * c, -k**2 * s, -k**3 * c, k**4 * s
        else:
            psi[:, col], d1[:, col], d2[:, col], d3[:, col], d4[:, col] = c, -k * s, -k**2 * c, k**3 * s, k**4 * c
    return psi, d1, d2, d3, d4


def ou_generator_actions(x):
    """psi, A psi and A^2 psi for the OU generator A f = -x f' + f''."""
    psi, d1, d2, d3, d4 = trig_dictionary(x)
    Apsi = -x[:, None] * d1 + d2
    A2psi = x[:, None] * d1 + (x[:, None] ** 2 - 2.0) * d2 - 2.0 * x[:, None] * d3 + d4
    return psi, Apsi, A2psi


def population(nquad=160):
    """Population matrices under N(0,1): G, H = <psi, A psi>, J = <psi, A^2 psi>, A_N = G^{-1} H, B_N = G^{-1} J."""
    x, w = normal_quadrature(nquad)
    psi, Apsi, A2psi = ou_generator_actions(x)
    cross = lambda L, R: L.T @ (R * w[:, None])
    G, H, J = cross(psi, psi), cross(psi, Apsi), cross(psi, A2psi)
    return x, w, G, H, J, solve(G, H, assume_a="pos"), solve(G, J, assume_a="pos")


def fit_slope(x, y):
    """Least-squares slope of log y against log x."""
    return float(np.polyfit(np.log(np.asarray(x, float)), np.log(np.asarray(y, float)), 1)[0])


def one_step_maps(A, B, dt):
    I = np.eye(A.shape[0])
    return I + dt * A, I + dt * A + 0.5 * dt**2 * (A @ A), I + dt * A + 0.5 * dt**2 * B

## Residual, leading operator, and lower bound

In [2]:
xq, wq, G, H, J, A, B = population()
N = A.shape[0]
I = np.eye(N)
D = B - A @ A
Theta = g_weighted_op_norm(D, G)

# G-orthonormal representation: A_N is self-adjoint, so its eigenbasis is G-orthonormal
Rch = sla.cholesky(0.5 * (G + G.T), lower=False)
Rinv = sla.solve_triangular(Rch, np.eye(N), lower=False)
As, Ds = Rch @ A @ Rinv, Rch @ D @ Rinv
lam, V = np.linalg.eigh(0.5 * (As + As.T))

ss = np.linspace(0.0, T, 4001)
L = 0.5 * np.trapezoid(np.array([expm((T - s) * A) @ D @ expm(s * A) for s in ss]), ss, axis=0)
normL = g_weighted_op_norm(L, G)
lower = 0.5 * T * max(math.exp(T * lam[j]) * float(V[:, j] @ Ds @ V[:, j]) for j in range(N))
ET = expm(T * A)

print(f"Theta_N = {Theta:.4f},  ||A_N|| = {g_weighted_op_norm(A, G):.4f},  ||B_N|| = {g_weighted_op_norm(B, G):.4f}")
print("eigenvalues of A_N:", np.round(lam, 4))
print(f"||L_{{N,T}}|| = {normL:.4f};  lower bound (3.11): (T/2) max_j e^(T lam_j) <v_j, D v_j> = {lower:.4f}")

# observable f = cos x (third dictionary function) and its exact image K^T f
psiq, *_ = trig_dictionary(xq)
f_coef = np.zeros(N); f_coef[2] = 1.0
KTf = math.exp(-0.5 * (1.0 - math.exp(-2.0 * T))) * np.cos(math.exp(-T) * xq)


def l2_err(coef):
    """L^2(rho) distance between sum_j coef_j psi_j and K^T cos x."""
    return float(math.sqrt(np.sum(wq * (psiq @ coef - KTf) ** 2)))


print(f"Galerkin error d_N(T, cos x) = ||e^{{T A_N}} Pi_N cos x - K^T cos x|| = {l2_err(ET @ f_coef):.4e}")

Theta_N = 3.6599,  ||A_N|| = 5.4399,  ||B_N|| = 32.1726
eigenvalues of A_N: [-5.4399 -4.9254 -2.0892 -1.2014  0.    ]
||L_{N,T}|| = 0.1077;  lower bound (3.11): (T/2) max_j e^(T lam_j) <v_j, D v_j> = 0.0815
Galerkin error d_N(T, cos x) = ||e^{T A_N} Pi_N cos x - K^T cos x|| = 9.5512e-03


## Table 4: time-stepping error, leading term, and error against $\mathcal K^T\cos x$

In [3]:
print("   n      tau    SDMD-1     SDMD-R     SDMD-2    rel.Q2  |  SDMD-1(f)  SDMD-R(f)  SDMD-2(f)")
table4 = []
for n in [10, 20, 40, 80, 160, 320]:
    tau = T / n
    Ps = [np.linalg.matrix_power(Q, n) for Q in one_step_maps(A, B, tau)]
    e1, eR, e2 = (g_weighted_op_norm(P - ET, G) for P in Ps)
    r2 = g_weighted_op_norm((Ps[2] - ET) / tau - L, G) / normL
    f1, fR, f2 = (l2_err(P @ f_coef) for P in Ps)
    table4.append((n, tau, e1, eR, e2, r2, f1, fR, f2))
    print(f"{n:4d}  {tau:.6f}  {e1:.2e}   {eR:.2e}   {e2:.2e}   {r2:.3f}  |  {f1:.3e}  {fR:.3e}  {f2:.3e}")
taus = [r[1] for r in table4]
for j, name in zip(range(2, 5), ["SDMD-1", "SDMD-R", "SDMD-2"]):
    print(f"slope over the three finest steps, {name}: {fit_slope(taus[-3:], [r[j] for r in table4[-3:]]):.2f}")

   n      tau    SDMD-1     SDMD-R     SDMD-2    rel.Q2  |  SDMD-1(f)  SDMD-R(f)  SDMD-2(f)
  10  0.100000  2.78e-02   2.40e-03   1.62e-02   0.574  |  1.750e-02  9.372e-03  8.974e-03
  20  0.050000  1.37e-02   5.10e-04   6.47e-03   0.224  |  1.262e-02  9.499e-03  9.299e-03
  40  0.025000  6.80e-03   1.22e-04   2.94e-03   0.102  |  1.076e-02  9.538e-03  9.442e-03
  80  0.012500  3.39e-03   3.00e-05   1.41e-03   0.049  |  1.006e-02  9.548e-03  9.501e-03
 160  0.006250  1.69e-03   7.42e-06   6.88e-04   0.024  |  9.778e-03  9.550e-03  9.527e-03
 320  0.003125  8.45e-04   1.85e-06   3.40e-04   0.012  |  9.658e-03  9.551e-03  9.539e-03
slope over the three finest steps, SDMD-1: 1.00
slope over the three finest steps, SDMD-R: 2.01
slope over the three finest steps, SDMD-2: 1.02


## Table 5: finite data at $\Delta t=0.05$

In [4]:
rng = np.random.default_rng(SEED)
dt, n = 0.05, 20
ms = [400, 800, 1600, 3200, 6400, 12800, 25600]
REPS = 25
print(f"errors against e^{{T A_N}} (G-weighted norm), mean [10%, 90%] over {REPS} repetitions; last columns: errors against K^T cos x")
print("     m     SDMD-1                     SDMD-R                     SDMD-2                 |  |Ahat-A|   |Bhat-B|  | SDMD-1(f)  SDMD-R(f)  SDMD-2(f)")
table5 = []
for m in ms:
    errs, ferr, dev = np.zeros((REPS, 3)), np.zeros((REPS, 3)), np.zeros((REPS, 2))
    for r in range(REPS):
        x = rng.standard_normal(m)
        psi, Apsi, A2psi = ou_generator_actions(x)
        Gh, Hh, Jh = psi.T @ psi / m, psi.T @ Apsi / m, psi.T @ A2psi / m
        Ah, Bh = np.linalg.solve(Gh, Hh), np.linalg.solve(Gh, Jh)
        Ps = [np.linalg.matrix_power(Q, n) for Q in one_step_maps(Ah, Bh, dt)]
        errs[r] = [g_weighted_op_norm(P - ET, G) for P in Ps]
        ferr[r] = [l2_err(P @ f_coef) for P in Ps]
        dev[r] = [g_weighted_op_norm(Ah - A, G), g_weighted_op_norm(Bh - B, G)]
    mean, lo, hi, fm = errs.mean(axis=0), np.percentile(errs, 10, axis=0), np.percentile(errs, 90, axis=0), ferr.mean(axis=0)
    table5.append((m, mean, lo, hi, dev.mean(axis=0), fm))
    cells = "  ".join(f"{mean[j]:.2e} [{lo[j]:.1e},{hi[j]:.1e}]" for j in range(3))
    print(f"{m:6d}  {cells}  |  {dev[:, 0].mean():.2e}  {dev[:, 1].mean():.2e}  | {fm[0]:.3e}  {fm[1]:.3e}  {fm[2]:.3e}")
for j, name in enumerate(["SDMD-1", "SDMD-R", "SDMD-2"]):
    print(f"slope in m over the five smallest sample sizes, {name}: {fit_slope(ms[:5], [t[1][j] for t in table5[:5]]):.2f}")

errors against e^{T A_N} (G-weighted norm), mean [10%, 90%] over 25 repetitions; last columns: errors against K^T cos x
     m     SDMD-1                     SDMD-R                     SDMD-2                 |  |Ahat-A|   |Bhat-B|  | SDMD-1(f)  SDMD-R(f)  SDMD-2(f)
   400  5.26e-02 [2.8e-02,8.6e-02]  5.14e-02 [2.7e-02,8.6e-02]  4.83e-02 [2.4e-02,8.2e-02]  |  6.50e-01  6.71e+00  | 1.384e-02  1.121e-02  1.076e-02
   800  3.53e-02 [1.9e-02,5.5e-02]  3.25e-02 [9.7e-03,5.1e-02]  2.94e-02 [9.9e-03,4.8e-02]  |  4.68e-01  5.19e+00  | 1.385e-02  1.152e-02  1.100e-02
  1600  2.53e-02 [1.6e-02,3.6e-02]  2.07e-02 [8.7e-03,3.6e-02]  2.01e-02 [1.0e-02,3.3e-02]  |  2.90e-01  3.16e+00  | 1.307e-02  1.015e-02  9.848e-03
  3200  2.28e-02 [1.5e-02,3.0e-02]  1.79e-02 [8.1e-03,2.9e-02]  1.73e-02 [8.4e-03,2.7e-02]  |  2.39e-01  2.60e+00  | 1.238e-02  9.538e-03  9.334e-03
  6400  1.84e-02 [1.4e-02,2.4e-02]  1.15e-02 [5.3e-03,1.9e-02]  1.23e-02 [6.9e-03,1.9e-02]  |  1.49e-01  1.52e+00  | 1.285e-02  9.688e-03 

 12800  1.54e-02 [1.3e-02,1.8e-02]  7.71e-03 [4.4e-03,1.1e-02]  9.75e-03 [6.9e-03,1.2e-02]  |  1.02e-01  1.13e+00  | 1.249e-02  9.483e-03  9.287e-03


 25600  1.53e-02 [1.4e-02,1.8e-02]  5.03e-03 [2.8e-03,7.3e-03]  7.46e-03 [5.5e-03,9.9e-03]  |  7.18e-02  7.77e-01  | 1.284e-02  9.623e-03  9.403e-03
slope in m over the five smallest sample sizes, SDMD-1: -0.37
slope in m over the five smallest sample sizes, SDMD-R: -0.52
slope in m over the five smallest sample sizes, SDMD-2: -0.47
